In [1]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    jaccard_score
)

In [2]:

# -------------------------
# Load data
# -------------------------
mask_clean      = np.load("../data/prompts/mask_clean.npy")          # (N,2,H,W) or similar
images_clean    = np.load("../data/prompts/images_clean.npy")        # (N,2,H,W) or similar
centroids_final = np.load("../data/prompts/centroids_final.npy", allow_pickle=True)
valid_idx       = np.load("../data/prompts/valid_idx.npy")

print(len(mask_clean), len(images_clean), len(centroids_final), len(valid_idx))
predictions_combined = np.load("../data/prompts/predictions_sam2.npy")

predictions_combined_b = predictions_combined.astype(bool)
gt = mask_clean.astype(bool)

2697 2697 2697 2697


In [3]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    jaccard_score
)

def evaluate_dataset(preds, gts, image_ids=None):
    """
    Compute metrics for all images in a dataset.

    Args:
        preds (list[np.ndarray]): predicted binary masks
        gts   (list[np.ndarray]): ground truth binary masks
        image_ids (list[str], optional): image identifiers

    Returns:
        dict: image_id -> metrics dict
    """
    if image_ids is None:
        image_ids = [i for i in range(len(preds))]

    results = {}

    for pred, gt, img_id in zip(preds, gts, image_ids):
        pred = pred.astype(bool)
        gt = gt.astype(bool)

        pred_f = pred.flatten().astype(int)
        gt_f = gt.flatten().astype(int)

        tp = np.logical_and(pred, gt).sum()
        tn = np.logical_and(~pred, ~gt).sum()
        fp = np.logical_and(pred, ~gt).sum()
        fn = np.logical_and(~pred, gt).sum()

        eps = 1e-8

        metrics = {
            "iou": tp / (tp + fp + fn + eps),
            "dice": (2 * tp) / (2 * tp + fp + fn + eps),
            "jaccard_distance": 1 - (tp / (tp + fp + fn + eps)),
            "pixel_accuracy": accuracy_score(gt_f, pred_f),
            "precision": precision_score(gt_f, pred_f, zero_division=0),
            "recall": recall_score(gt_f, pred_f, zero_division=0),
            "f1": f1_score(gt_f, pred_f, zero_division=0),
            "specificity": tn / (tn + fp + eps),
            "false_positive_rate": fp / (fp + tn + eps),
            "false_negative_rate": fn / (fn + tp + eps),
            "volume_similarity": 1 - abs(pred.sum() - gt.sum()) /
                                 (pred.sum() + gt.sum() + eps),
            "sklearn_jaccard": jaccard_score(gt_f, pred_f, zero_division=0),
        }

        results[img_id] = metrics

    return results



In [4]:
results_0 = evaluate_dataset(predictions_combined_b[:, 0], gt[:,0])
results_1 = evaluate_dataset(predictions_combined_b[:, 0], gt[:,0])
q_thresh = 0.25

good_images = {
    img_id: m
    for img_id, m in results_0.items()
    if (m["false_positive_rate"]) < q_thresh
}
bad_images = {
    img_id: m
    for img_id, m in results_1.items()
    if (m["false_positive_rate"]) >= q_thresh
}

print(len(good_images))
print(len(bad_images))

2674
23


In [7]:
import numpy as np

q_thresh = 0.25

# Evaluate both channels
results_0 = evaluate_dataset(predictions_combined_b[:, 0], gt[:, 0])
results_1 = evaluate_dataset(predictions_combined_b[:, 1], gt[:, 1])

# Select only images where both channels are "good"
good_images_both = {
    img_id: (m0, m1)
    for img_id, (m0, m1) in zip(range(len(gt)), zip(results_0.values(), results_1.values()))
    if m0["false_positive_rate"] < q_thresh and m1["false_positive_rate"] < q_thresh
}

# Get the indices of the good images
valid_idx = np.array(list(good_images_both.keys()))

# Filter images and predictions
images_cleaned_prompted = images_clean[valid_idx]                # shape (N_good, 2, H, W)
predictions_clean = predictions_combined_b[valid_idx]  # shape (N_good, 2, H, W)

# Save results
np.save("../data/prompted_outputs/images_cleaned_prompted.npy", images_cleaned_prompted)
np.save("../data/prompted_outputs/valid_idx_prompted.npy", valid_idx)
np.save("../data/prompted_outputs/predictions_sam2_cleaned.npy", predictions_clean)

print(f"Kept {len(valid_idx)} images where both channels are good.")


Kept 2551 images where both channels are good.


In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
import numpy as np
from skimage import exposure

def normalize_contrast(img, pmin=2, pmax=98):
    """Stretch contrast to [pmin,pmax] percentiles."""
    pmin_val, pmax_val = np.percentile(img, (pmin, pmax))
    return exposure.rescale_intensity(img, in_range=(pmin_val, pmax_val), out_range=(0, 255)).astype(np.uint8)

def visualize_image(idx):
    plt.figure(figsize=(24, 10))
    plt.tight_layout(pad=2.0)

    # ----------------------------
    # Channel 0
    # ----------------------------
    img0_raw = images_clean[idx, 0]
    img0_norm = normalize_contrast(img0_raw)
    
    # Image with centroids
    img0_rgb = np.stack([img0_norm] * 3, axis=-1)
    for xy in centroids_final[idx][0]:
        x, y = int(round(xy[0])), int(round(xy[1]))
        if 0 <= y < img0_rgb.shape[0] and 0 <= x < img0_rgb.shape[1]:
            img0_rgb[y, x] = [0, 255, 0]  # bright green centroid

    # Predicted overlay
    ov0 = img0_rgb.copy()
    pred_mask0 = predictions_ch0[idx] > 0
    ov0[pred_mask0] = [255, 0, 0]  # red mask overlay

    # Ground truth overlay
    gt_mask0 = mask_clean[idx, 0] > 0
    gt0 = img0_rgb.copy()
    gt0[gt_mask0] = [255, 255, 0]  # yellow GT mask

    # Plot CH0
    plt.subplot(2, 5, 1)
    plt.imshow(img0_raw, cmap='gray')
    plt.title("CH0 Raw")
    plt.axis('off')

    plt.subplot(2, 5, 2)
    plt.imshow(img0_norm, cmap='gray')
    plt.title("CH0 Normalized")
    plt.axis('off')

    plt.subplot(2, 5, 3)
    plt.imshow(img0_rgb)
    plt.title("CH0 Image + Centroids")
    plt.axis('off')

    plt.subplot(2, 5, 4)
    plt.imshow(gt0)
    plt.title("CH0 GT Mask Overlay")
    plt.axis('off')

    plt.subplot(2, 5, 5)
    plt.imshow(ov0)
    plt.title("CH0 Pred Mask Overlay")
    plt.axis('off')

    # ----------------------------
    # Channel 1
    # ----------------------------
    img1_raw = images_clean[idx, 1]
    img1_norm = normalize_contrast(img1_raw)
    
    # Image with centroids
    img1_rgb = np.stack([img1_norm] * 3, axis=-1)
    for xy in centroids_final[idx][1]:
        x, y = int(round(xy[0])), int(round(xy[1]))
        if 0 <= y < img1_rgb.shape[0] and 0 <= x < img1_rgb.shape[1]:
            img1_rgb[y, x] = [0, 255, 0]  # bright green centroid

    # Predicted overlay
    ov1 = img1_rgb.copy()
    pred_mask1 = predictions_ch1[idx] > 0
    ov1[pred_mask1] = [255, 0, 255]  # magenta mask overlay

    # Ground truth overlay
    gt_mask1 = mask_clean[idx, 1] > 0
    gt1 = img1_rgb.copy()
    gt1[gt_mask1] = [255, 255, 0]  # yellow GT mask

    # Plot CH1
    plt.subplot(2, 5, 6)
    plt.imshow(img1_raw, cmap='gray')
    plt.title("CH1 Raw")
    plt.axis('off')

    plt.subplot(2, 5, 7)
    plt.imshow(img1_norm, cmap='gray')
    plt.title("CH1 Normalized")
    plt.axis('off')

    plt.subplot(2, 5, 8)
    plt.imshow(img1_rgb)
    plt.title("CH1 Image + Centroids")
    plt.axis('off')

    plt.subplot(2, 5, 9)
    plt.imshow(gt1)
    plt.title("CH1 GT Mask Overlay")
    plt.axis('off')

    plt.subplot(2, 5, 10)
    plt.imshow(ov1)
    plt.title("CH1 Pred Mask Overlay")
    plt.axis('off')

    plt.suptitle(f"Image {idx}: SAM2 Segmentation Results", fontsize=16, y=0.98)
    plt.show()
    print(df.iloc[idx])
predictions_ch0 = predictions_combined_b[:, 0]
predictions_ch1 = predictions_combined_b[:, 1]
# Interactive slider (uncomment to use full dataset)
interact(
    visualize_image,
    idx=IntSlider(min=0, max=min(1000, len(images_clean)-1), step=1, value=0)
)


interactive(children=(IntSlider(value=0, description='idx', max=1000), Output()), _dom_classes=('widget-intera…

<function __main__.visualize_image(idx)>